### Create dataset

In [4]:
import pandas as pd
import numpy as np

np.random.seed(42)

# -----------------------------
# CONFIG
# -----------------------------
N_ORDERS = 12000

date_range = pd.date_range(start="2024-01-01", end="2025-06-30")

categories = {
    "Electronics": {"margin": 0.25, "discount_bias": 0.6},
    "Beauty": {"margin": 0.65, "discount_bias": 0.2},
    "Supplements": {"margin": 0.55, "discount_bias": 0.3},
    "Home": {"margin": 0.45, "discount_bias": 0.4},
    "Accessories": {"margin": 0.70, "discount_bias": 0.15}
}

channels = ["Email", "Organic", "Google", "Facebook", "Referral"]

# -----------------------------
# PRODUCT LIST
# -----------------------------
products = []

for i in range(1, 51):
    category = np.random.choice(list(categories.keys()))
    base_price = np.random.uniform(15, 200)

    products.append({
        "product_id": f"P{i:03d}",
        "product_name": f"Product {i}",
        "category": category,
        "base_price": base_price
    })

products_df = pd.DataFrame(products)

# -----------------------------
# HELPER FUNCTIONS
# -----------------------------
def get_discount(category, channel):
    bias = categories[category]["discount_bias"]

    base_discount = np.random.beta(2, 5) * 0.5

    if channel == "Facebook":
        base_discount *= 1.3
    elif channel == "Email":
        base_discount *= 0.8

    discount = min(base_discount + bias * 0.2, 0.6)

    return discount

def seasonality_multiplier(date):
    month = date.month

    if month in [11, 12]:
        return 1.3
    elif month in [6, 7, 8]:
        return 0.9
    else:
        return 1.0

# -----------------------------
# GENERATE ORDERS
# -----------------------------
rows = []

for i in range(N_ORDERS):
    product = products_df.sample(1).iloc[0]

    date = np.random.choice(date_range)
    channel = np.random.choice(channels)

    category = product["category"]
    base_price = product["base_price"]

        # Discount influences demand
    if discount_pct > 0.4:
        units = np.random.choice([1, 2, 3], p=[0.4, 0.4, 0.2])
    elif discount_pct > 0.2:
        units = np.random.choice([1, 2, 3], p=[0.5, 0.3, 0.2])
    else:
        units = np.random.choice([1, 2, 3], p=[0.8, 0.15, 0.05])

    # Category sensitivity
    if category == "Electronics":
        units *= np.random.choice([1, 1.2], p=[0.7, 0.3])
    elif category == "Accessories":
        units *= np.random.choice([1, 1.1], p=[0.9, 0.1])

    units = int(round(units))
    
    discount_pct = get_discount(category, channel)

    season_mult = seasonality_multiplier(pd.Timestamp(date))

    adjusted_price = base_price * season_mult

    final_price = adjusted_price * (1 - discount_pct)

    revenue = final_price * units

    margin = categories[category]["margin"]

    cogs = adjusted_price * (1 - margin) * units

    profit = revenue - cogs

    profit_margin_pct = profit / revenue if revenue > 0 else 0

    rows.append({
        "order_id": f"O{i+1:06d}",
        "order_date": date,
        "product_id": product["product_id"],
        "product_name": product["product_name"],
        "category": category,
        "channel": channel,
        "units": units,
        "base_price": round(base_price, 2),
        "discount_pct": round(discount_pct, 3),
        "final_price": round(final_price, 2),
        "revenue": round(revenue, 2),
        "cogs": round(cogs, 2),
        "profit": round(profit, 2),
        "profit_margin_pct": round(profit_margin_pct, 3)
    })

df = pd.DataFrame(rows)

df.to_csv("pricing_orders.csv", index=False)

df.head()

,order_id,order_date,product_id,product_name,category,channel,units,base_price,discount_pct,final_price,revenue,cogs,profit,profit_margin_pct
0,O000001,2024-09-15,P033,Product 33,Home,Google,1,23.37,0.194,18.83,18.83,12.85,5.97,0.317
1,O000002,2024-05-26,P013,Product 13,Supplements,Facebook,1,128.19,0.355,82.66,82.66,57.69,24.98,0.302
2,O000003,2024-06-09,P046,Product 46,Electronics,Facebook,1,130.31,0.292,83.03,83.03,87.96,-4.93,-0.059
3,O000004,2024-11-18,P010,Product 10,Accessories,Organic,2,129.23,0.230,129.31,258.62,100.80,157.82,0.610
4,O000005,2024-09-25,P003,Product 3,Accessories,Organic,2,43.86,0.092,39.85,79.69,26.32,53.37,0.670


### Check data

In [5]:
df.describe()

df.groupby("category")[["revenue", "profit"]].sum()

df.groupby("channel")[["revenue", "profit"]].mean()

df["discount_pct"].describe()

count    12000.000000
mean         0.205270
std          0.087771
min          0.031000
25%          0.139000
50%          0.193000
75%          0.260000
max          0.600000
Name: discount_pct, dtype: float64

In [7]:
df["units"].value_counts()

units
1    7830
2    2691
3    1452
4      27
Name: count, dtype: int64

### Discount vs Profit analysis

In [8]:
df.groupby(pd.cut(df["discount_pct"], bins=5))[
    ["revenue", "profit"]
].mean()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_5968\2565211690.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.cut(df["discount_pct"], bins=5))[


,revenue,profit
discount_pct,,
"(0.0304, 0.145]",133.795508,74.449084
"(0.145, 0.259]",118.166954,52.040524
"(0.259, 0.372]",103.636435,32.476359
"(0.372, 0.486]",92.051844,15.923307
"(0.486, 0.6]",57.710213,-7.608936


### Category Profitability

In [9]:
df.groupby("category")[["revenue", "profit", "profit_margin_pct"]].mean()

,revenue,profit,profit_margin_pct
category,,,
Accessories,122.200416,77.514940,0.630816
Beauty,127.158815,72.490649,0.564028
Electronics,134.030843,-2.644491,-0.038275
Home,99.652283,29.076438,0.279809
Supplements,121.948373,52.588550,0.426059


### Channel + discount interaction

In [10]:
df.groupby("channel")[["discount_pct", "profit_margin_pct"]].mean()

,discount_pct,profit_margin_pct
channel,,
Email,0.173393,0.456286
Facebook,0.243433,0.393802
Google,0.201971,0.435315
Organic,0.202519,0.435838
Referral,0.204080,0.429741


### High product discounts

In [11]:
df[df["discount_pct"] > 0.4].groupby("product_name")[
    ["revenue", "profit"]
].sum().sort_values("profit")

,revenue,profit
product_name,,
Product 40,2109.48,-755.79
Product 46,1193.09,-370.63
Product 38,810.11,-310.33
Product 25,493.49,-22.52
Product 28,570.24,-17.26
Product 24,687.86,-12.20
Product 33,243.57,-8.32
Product 6,235.49,-1.46
Product 7,7.71,-0.75


### Profit per unit vs discount

In [12]:
df["profit_per_unit"] = df["profit"] / df["units"]

df.groupby(pd.cut(df["discount_pct"], bins=5))[
    "profit_per_unit"
].mean()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_5968\3844134361.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.cut(df["discount_pct"], bins=5))[


discount_pct
(0.0304, 0.145]    50.585996
(0.145, 0.259]     35.515825
(0.259, 0.372]     21.584886
(0.372, 0.486]     10.886204
(0.486, 0.6]       -5.604929
Name: profit_per_unit, dtype: float64

### Export data to Power BI

In [13]:
df.to_csv("shopify_pricing_analysis.csv", index=False)